# Cross-Chip Consistency Model

**CRISP-DM Phase 4 — Modeling**  
Predicting cross-chip SD variability (`cross_chip_std`) from waveform parameters.  
Target: find which V, F_r, dt2 settings produce the most consistent SD output across all 30 chips for a given color and coverage level.

See `documentation/model-2-cross-chip-consistency.md` for design decisions and trade-offs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available — skipping. Run: brew install libomp')

## 1. Load and prepare
*CRISP-DM Phase 3 — Data Preparation*

Each row in the extended file is one waveform condition × one color, with nozzle values for all 30 chips in columns.  
We assign the color label (rows appear in fixed order M, C, Y, K per condition) and compute the target:  
**`cross_chip_std`** = standard deviation of the 30 per-chip mean SD values.  
This measures how consistently all chips respond to the same waveform setting.

In [ ]:
df = pd.read_parquet('../../data/new/extended-rows.parquet 2')
value_cols = [c for c in df.columns if 'Value' in c]
cond_cols  = ['V', 'F_r', 'dt2', 'Coverage#']

# Rows per condition are always ordered M, C, Y, K
df['color_idx'] = df.groupby(cond_cols).cumcount()
df['Color$']    = df['color_idx'].map({0: 'M', 1: 'C', 2: 'Y', 3: 'K'})

# Per-chip mean SD, then cross-chip std and mean
chip_means = pd.DataFrame(index=df.index)
for chip in range(1, 31):
    cols = [c for c in value_cols if c.startswith(f'{float(chip)}_')]
    chip_means[f'chip_{chip}'] = df[cols].mean(axis=1)

df['cross_chip_std']   = chip_means.std(axis=1)
df['cross_chip_mean'] = chip_means.mean(axis=1)

print(f'Rows: {len(df):,}')
print(f'cross_chip_std   range: {df["cross_chip_std"].min():.5f} – {df["cross_chip_std"].max():.5f}')
print(f'cross_chip_mean range: {df["cross_chip_mean"].min():.5f} – {df["cross_chip_mean"].max():.5f}')

In [ ]:
df.columns

## 2. Feature engineering
*CRISP-DM Phase 3 — Data Preparation*

Same features as model 1: raw waveform params + interaction and polynomial terms.  
Color and Coverage are inputs — they are fixed by the print job, not tunable.

In [ ]:
color_dummies = pd.get_dummies(df['Color$'], prefix='color', drop_first=False)

base = df[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base['V_x_Fr']         = df['V'] * df['F_r']
base['dt2_x_coverage'] = df['dt2'] * df['Coverage#']
base['V_sq']           = df['V'] ** 2
base['Fr_sq']          = df['F_r'] ** 2

X = pd.concat([base, color_dummies], axis=1)
y = df['cross_chip_std']

print(f'Features: {X.columns.tolist()}')
print(f'Samples:  {len(X):,}')

## 3. Train / test split
*CRISP-DM Phase 4 — Modeling: Generate Test Design*

Random 80/20 split stratified by Color.  
No HeadIdx# available in this dataset — chip identity is encoded in the columns, not rows.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['Color$']
)

print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_test):,} rows')

## 4. Train and evaluate models
*CRISP-DM Phase 4 — Modeling: Build Model*

Three model candidates are trained and compared. The best is selected automatically by R² on the test set.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5,
                                     random_state=42, verbosity=0)

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'R2':     r2_score(y_test, y_pred),
        'MAE':    mean_absolute_error(y_test, y_pred),
        'model':  model,
        'y_pred': y_pred,
    }

summary = pd.DataFrame({k: {'R²': v['R2'], 'MAE': v['MAE']} for k, v in results.items()}).T
print(summary.round(4))

## 5. Predicted vs actual
*CRISP-DM Phase 5 — Evaluation*

Visual check: points close to the diagonal mean the model predicts well. Scatter indicates where it struggles.

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.3, s=5)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual cross_chip_std')
    ax.set_ylabel('Predicted cross_chip_std')
    ax.set_title(f'{name}\nR²={res["R2"]:.3f}  MAE={res["MAE"]:.5f}')

plt.tight_layout()
plt.show()

## 6. Feature importance (Random Forest)
*CRISP-DM Phase 5 — Evaluation: Assess Model*

This is a **model validation step** — checking whether the model learned meaningful patterns by comparing against the earlier coverage analysis.

### What to look for
If this model's feature ranking differs from model 1, it suggests that the drivers of **cross-chip consistency** are not the same as the drivers of **within-chip uniformity** — which would be an important finding for Canon.

Compare the ranking here against model 1:  
- Model 1 top drivers: Coverage# (27%), Color (28% combined), V/V² (22% combined), F_r (15% combined), dt2 (0.8%)  
- If Coverage# and V are still dominant here, both problems are driven by the same parameters  
- If F_r or dt2 rank higher here, cross-chip consistency may require a different tuning strategy than within-chip uniformity

In [ ]:
rf = results['Random Forest']['model']
importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature importance — Random Forest')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()

## 7. Optimal waveform settings per (Color, Coverage)
*CRISP-DM Phase 5 — Evaluation: Determine next steps*

For each (Color, Coverage) combination, find the V, F_r, dt2 settings that minimise cross-chip variability.  
Coverage is fixed by the desired ink density — this table tells you the best tunable settings for each scenario.

In [ ]:
best_name  = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}  (R²={results[best_name]["R2"]:.4f})')

param_cols = ['Color$', 'V', 'F_r', 'dt2', 'Coverage#']
grid = df[param_cols + ['cross_chip_mean']].drop_duplicates(subset=param_cols).reset_index(drop=True).copy()

color_dummies_grid = pd.get_dummies(grid['Color$'], prefix='color', drop_first=False)
base_grid = grid[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base_grid['V_x_Fr']         = grid['V'].values * grid['F_r'].values
base_grid['dt2_x_coverage'] = grid['dt2'].values * grid['Coverage#'].values
base_grid['V_sq']           = grid['V'].values ** 2
base_grid['Fr_sq']          = grid['F_r'].values ** 2

X_grid = pd.concat([base_grid, color_dummies_grid], axis=1)
X_grid = X_grid.reindex(columns=X_train.columns, fill_value=0)

grid['predicted_cross_chip_std'] = best_model.predict(X_grid)

best_per_coverage = (
    grid.sort_values('predicted_cross_chip_std')
    .groupby(['Color$', 'Coverage#'], group_keys=False)
    .head(1)
    .sort_values(['Color$', 'Coverage#'])
)
print()
print(best_per_coverage[['Color$', 'Coverage#', 'V', 'F_r', 'dt2', 'cross_chip_mean', 'predicted_cross_chip_std']].to_string(index=False))

## 8. Manual prediction
*CRISP-DM Phase 5 — Evaluation*

Change the values below and run the cell to get a predicted `cross_chip_std` for any waveform setting.

In [ ]:
# --- change these values ---
V        = 20.0   # options: 20, 22, 24, 26, 28, 30
F_r      = 1.02   # options: 1.02, 1.08, 1.09, ..., 1.379
dt2      = -2000  # options: -1100, -900, -700, -500, -300
coverage = 2.0    # options: 2 – 31
color    = 'M'    # options: C, M, Y, K
# ---------------------------

custom = pd.DataFrame([{'V': V, 'F_r': F_r, 'dt2': dt2, 'Coverage#': coverage, 'Color$': color}])

color_dummies = pd.get_dummies(custom['Color$'], prefix='color', drop_first=False)
base = custom[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base['V_x_Fr']         = custom['V'] * custom['F_r']
base['dt2_x_coverage'] = custom['dt2'] * custom['Coverage#']
base['V_sq']           = custom['V'] ** 2
base['Fr_sq']          = custom['F_r'] ** 2

X_custom = pd.concat([base, color_dummies], axis=1)
X_custom = X_custom.reindex(columns=X_train.columns, fill_value=0)

pred = best_model.predict(X_custom)[0]

# Look up actual value if this exact combination exists in the data
match = df[
    (df['V'] == V) & (df['F_r'] == F_r) & (df['dt2'] == dt2) &
    (df['Coverage#'] == coverage) & (df['Color$'] == color)
]

print(f'Settings: V={V}, F_r={F_r}, dt2={dt2}, Coverage={coverage}, Color={color}')
print(f'Predicted cross_chip_std: {pred:.6f}')
if len(match) > 0:
    actual = match['cross_chip_std'].values[0]
    actual_mean = match['cross_chip_mean'].values[0]
    print(f'Actual    cross_chip_std: {actual:.6f}  (error: {abs(pred - actual):.6f})')
    print(f'Actual    cross_chip_mean (SD level): {actual_mean:.6f}')
else:
    print('Actual: not in dataset — prediction only')
print(f'Lower is better. Best in dataset: {df["cross_chip_std"].min():.6f}')

## 9. Find best settings for a target SD level
*CRISP-DM Phase 5 — Evaluation*

Find the V, F_r, dt2 settings that:  
1. Produce a `cross_chip_mean` (average SD across all 30 chips) close to your **target**  
2. Among those, return the ones with the lowest **cross_chip_std** — most consistent chips

This is the most practical use of model 2: *get the right ink density AND make all 30 chips consistent at that level.*

In [ ]:
# --- change these values ---
target_sd_mean = 0.35   # desired average SD level across all chips
tolerance      = 0.02   # how close cross_chip_mean must be to the target (±)
color          = 'C'    # options: C, M, Y, K
# ---------------------------

filtered = df[
    (df['Color$'] == color) &
    (df['cross_chip_mean'] >= target_sd_mean - tolerance) &
    (df['cross_chip_mean'] <= target_sd_mean + tolerance)
]

if len(filtered) == 0:
    print(f'No conditions found for Color={color} with cross_chip_mean between {target_sd_mean - tolerance:.3f} and {target_sd_mean + tolerance:.3f}')
    print(f'cross_chip_mean range for {color}: {df[df["Color$"]==color]["cross_chip_mean"].min():.3f} – {df[df["Color$"]==color]["cross_chip_mean"].max():.3f}')
else:
    best = (
        filtered.sort_values('cross_chip_std')
        .groupby(['V', 'F_r', 'dt2'], group_keys=False)
        .head(1)
        .sort_values('cross_chip_std')
        .head(10)
    )
    print(f'Color={color} | Target cross_chip_mean={target_sd_mean} ± {tolerance} | {len(filtered)} matching conditions found')
    print()
    print(best[['Color$', 'V', 'F_r', 'dt2', 'Coverage#', 'cross_chip_mean', 'cross_chip_std']].to_string(index=False))